# Hackathon “Hack the Promoter”

**iGEM Warsaw 2026**, 29th August 2026

## 1. Introduction

The hackathon deals with problems at the intersection of computer science and molecular biology, in particular the analysis and interpretation of the information stored in DNA.

### 1.1 DNA

DNA is the fundamental carrier of genetic information in living organisms. It is a molecule built from repeating nucleotides, denoted by the letters **A**, **T**, **C** and **G**, whose order encodes biological information. Even a single change in the nucleotide sequence can have catastrophic consequences for a cell or for an entire organism.

### 1.2 Gene

A gene is a fragment of DNA containing the information needed to make a particular product, most often a protein. Proteins serve many functions in the cell: they enable biochemical reactions, act as building material and regulate cellular processes.

The cell cannot, however, make a protein directly from DNA. It must first transcribe DNA into mRNA in the process of **transcription**, and then translate mRNA into protein in the process of **translation**. mRNA, like DNA, is built from nucleotides denoted in bioinformatics by the same letters[^1]. A protein is built from amino acids.

[^1]: In reality these are the letters **A**, **U**, **C** and **G** in mRNA.

### 1.3 Promoter

A promoter is a region of DNA located upstream of the coding sequence of a gene and responsible for initiating transcription. It does not encode a protein, but plays a regulatory role and is part of the gene. The promoter sequence:

- allows the cellular machinery that carries out transcription to bind to the DNA,
- marks the place where the gene begins,
- influences the level of expression.

Promoters can be strong or weak. A strong promoter does not need many factors to initiate transcription of a gene, and that is exactly how we define promoter strength here: as the ability to initiate transcription efficiently. High-strength promoters lead to intensive transcription and high mRNA levels, while weak ones result in lower expression. In biotechnology, control over the level of transcription determines the efficiency of production processes, for example in the production of yeast proteins used in making alcohol.

### 1.4 *Trichoderma atroviride*

*Trichoderma atroviride* is a fungal species of biotechnological and agricultural importance. It is used because it:

- produces proteins that break down the cell-wall components of other organisms,
- serves as a biological crop-protection agent, inhibiting the growth of pathogens,
- is relatively easy to culture and amenable to genetic modification.

*T. atroviride* produces a compound abbreviated as 6PP. It has a characteristic coconut smell and biological activity: it acts as an antifungal and stimulates plant growth. 6PP therefore matters in basic research, in industry and in agriculture.

The goal of the hackathon task is to **design a strong promoter for the `pks1` gene, responsible for the biosynthesis of 6PP, in *Trichoderma atroviride* strain P1**. The scoring model was trained on promoters from the whole *Trichoderma* genus, but the task concerns this one gene and this one strain. Successfully designing a strong promoter translates directly into increased production of 6PP, and through that into the efficiency of the industrial and agricultural processes that use this compound.

## 2. Task specification

**Task:** design promoter sequences that increase transcription of the **`pks1`** gene in *Trichoderma atroviride* strain **P1**, in the category of **biotic antagonism**, that is, competition with other microorganisms.

**The starting point is given.** The natural `pks1` promoter, 800 bp long and aligned to the transcription start site, is obtained from the API (`GET /dziki`). The task is about improving it, although you may submit any sequence of 800 bp.

**Sequence requirements.** Every sequence must:

- be **exactly 800 bp**,
- consist of the characters `ACGTN`,
- contain at most **10% `N`**,
- be **unique** within the file.

Those are all the requirements. Sequences that do not meet them are skipped, and the server response reports their identifiers. The rationale for the `N` threshold is in section 3.3.

**Upload format:** a **FASTA** file. The first 100 sequences that pass the filters are scored. Any excess is ignored, so sending more than 100 sequences gains nothing.

**Tools.** Every team has access at all times to two model-based tools: the **Judge** (`/sedzia`), which for a pair of sequences indicates the stronger one, and the **Navigator** (`/nawigator`), which shows the structure of a sequence and proposes edits. Neither returns a numerical strength value; section 3.1 explains why. The score itself is issued by a separate model, the **Oracle**, which is not available to participants. Example usage with explanation is presented in this notebook.

**Evaluation.** Teams may upload a file multiple times. The system scores a submission automatically and updates the team's result, with the **best** score counting, not the last one. The final grade consists of:

- **25%** of the points for the average score of the 10 best sequences out of the pool of 100, graded by rank. Teams do not know which sequences ended up in that top ten.
- **25%** of the points for the average score of all 100 sequences, graded by rank.
- **50%** of the points for presenting the solution to the Jury. What is assessed: the analytical approach, creativity in optimisation, biological awareness, the description of the methodology (whether the method is general, what its limitations are, how it could be improved) and the quality of the answers to the Jury members' questions.

**The ranking** can be checked at any time during the hackathon with no limit, via `GET /ranking`, most easily by running the last cell of the notebook, which prints a ready-made table. **There is no separate web page with a scoreboard**; the API is the only interface. The table shows only the points in both categories, without the raw Oracle scores.

## 3. Access to the scoring model (API)

The model is exposed as an HTTP service at:

```
https://hyppe.futura.foundation
```

Authentication uses the `X-API-Key` header. Each participant receives an individual key. The model weights are not shared; the service returns only the result of calling it.

### 3.1 Three roles of the model, two of them available

| model | available to | what it does |
|---|---|---|
| **Judge** (`sedzia`) | participants | for a pair of sequences, indicates the stronger one |
| **Navigator** (`nawigator`) | participants | map of the sequence and proposed edits |
| **Oracle** (`wyrocznia`) | Jury only | issues the point score for submissions |

The Judge and the Navigator work without access to the gene and species context; they judge the sequence itself. The Oracle judges in the context of the `pks1` gene.

### 3.2 Endpoints

| Method | Path | Description | Limit |
|---|---|---|---|
| `GET` | `/dziki` | the starting `pks1` promoter, 800 bp | no limit |
| `POST` | `/sedzia` | a pair of sequences, response names the stronger one | 600/min |
| `POST` | `/nawigator/mapa` | position map: reconstruction, layers, changes under P1 | 600/min |
| `POST` | `/nawigator/edycje` | proposed edits by changing latent codes | 600/min |
| `POST` | `/wgraj` | submit a FASTA file for scoring | once per 5 min |
| `GET` | `/ranking` | scoreboard, points only | no limit |
| `GET` | `/me` | key status, limits, daily usage | no limit |

The body of `POST` requests is **JSON**, not `multipart`. The FASTA file is sent as a string in the `fasta` field. A ready-made Python client, a single `call` function with no dependencies beyond the standard library and with automatic retries, is in the practical part of this notebook.

### 3.3 Submission format

A **FASTA** file in UTF-8. The server scores the submission and does not review the sequences. Before scoring it applies only the technically necessary filters, in this order:

1. skips sequences whose length is other than **800 bp**, because the model input has a fixed size,
2. skips duplicates, keeping the first occurrence,
3. skips sequences with characters outside `ACGTN`, because the remaining characters have no representation in the encoding,
4. skips sequences with a share of `N` above **10%**, for the reason described below,
5. scores the first 100 remaining sequences.

The response reports how many sequences were read, how many were skipped and for what reason, together with the **identifiers** of the skipped ones, so that the file can be corrected.

**Where the `N` threshold comes from.** An `N` position enters the model as 0.25 in each of the four one-hot encoding channels, rather than as 1.0 in one of them. A sequence dense in `N` therefore feeds the model a distribution it did not see during training, and its score stops measuring the sequence. This is a numerical limitation arising from the input encoding.

A submission may be repeated many times. The **best** result of a team counts for the ranking, not the last one. The interval between submissions from one key is 5 minutes; the Jury has no interval.

### 3.4 Automatic scoring

$$\mathrm{TOP10} = \frac{\sum \text{10 best scores}}{10}, \qquad \mathrm{ALL100} = \frac{\sum \text{scores of all sequences}}{100}$$

The divisors are **fixed**. A file containing fewer than 100 sequences is not rejected, but receives a proportionally lower score in the ALL100 category. Submitting 50 sequences means in practice half of the possible points in that category.

Both categories are converted to points by rank: first place receives 10 points, last place 0, linearly in between. Points are awarded only to teams that uploaded anything. Ties are broken by upload time, that is, the team that reached a given result earlier ranks higher.

These two categories make up 25% and 25% of the final grade, and the remaining 50% is the assessment of the presentation before the Jury.

### 3.5 Limits

| what | value |
|---|---|
| Judge | 600 requests/min per key |
| Navigator (`/mapa`, `/edycje`) | 600 requests/min per key, each separately |
| other `POST` endpoints | 240 requests/min per key |
| `/me`, `/ranking`, `/dziki` | no limit |
| file submission | once per 5 minutes per key |
| FASTA file size | up to 2,000,000 characters |

The limits are deliberately high. They are not there to make you economise, but to stop one team from taking the whole queue.

## 4. Why is this topic interesting?

Many substances used by people are produced thanks to microorganisms that carry genes responsible for the biosynthesis of valuable compounds. Efficient production of such metabolites requires a strong promoter. Designing promoters with higher activity than those occurring naturally opens up new possibilities, in particular in the development of therapeutics.

One example is insulin, whose production using modified microorganisms made efficient and safe manufacturing at industrial scale possible, and through that the wide availability and high quality of diabetes therapy. Refining genetic tools such as promoters may in the future allow similarly efficient production of other therapeutic compounds. That is an opportunity for large-scale industry and, in this case, for the development of biological crop-protection agents.

# Practical part: the API step by step

Replace the key in the next cell. The rest works unchanged.

## 5.1 Configuration

The only line to replace is `KEY`. The `User-Agent` header is
needed because the client uses `urllib`.

In [ ]:
KEY = "YOUR_KEY" # CHANGE IT
URL = "https://hyppe.futura.foundation"

import json, statistics, urllib.error, urllib.request

UA = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/128.0 Safari/537.36")


def call(path, data=None, timeout=900, tries=6):
    """(http_code, response). `data` != None -> POST with JSON.
    """
    import time
    for n in range(tries):
        req = urllib.request.Request(URL.rstrip('/') + path)
        req.add_header('X-API-Key', KEY)
        req.add_header('User-Agent', UA)
        if data is not None:
            req.add_header('Content-Type', 'application/json')
            req.data = json.dumps(data).encode()
        try:
            with urllib.request.urlopen(req, timeout=timeout) as o:
                return o.status, json.loads(o.read().decode())
        except urllib.error.HTTPError as e:
            body = e.read().decode('utf-8', 'replace')
            retryable = (e.code == 503
                         or (e.code == 429 and path != '/wgraj'))
            if not retryable or n == tries - 1:
                return e.code, body
            wait = float(e.headers.get('Retry-After') or 0) or 0.4 * 2 ** n
            time.sleep(min(wait, 8.0))
        except urllib.error.URLError as e:
            if n == tries - 1:
                return 0, 'network: %s' % e
            time.sleep(0.4 * 2 ** n)


code_, me = call('/me')
assert code_ == 200, ('bad key or missing User-Agent', me)
print('team       :', me['druzyna'], '|', me['uczestnik'])
print('key valid  :', me['wazny_do'], '(%.1f h)' % (me['wazny_jeszcze_s'] / 3600)
      if me.get('wazny_jeszcze_s') else 'key without expiry')
print('limits/min :', me['limity_na_minute'])
print('upload in  :', me['zgloszenie_mozliwe_za_s'], 's')

## 5.2 The starting promoter

`GET /dziki` returns an 800 bp sequence, its length and a checksum.

In [ ]:
code_, d = call('/dziki')
assert code_ == 200, d
WILD = d['sekwencja']
print('%s | gene %s | %s' % (d['nazwa'], d['gen'], d['genom']))
print('length %d bp | sha256[:12] = %s' % (d['dlugosc'], d['sha256_12']))
assert len(WILD) == 800

## 5.3 Navigator: map of the sequence

`POST /nawigator/mapa` describes every position of the sequence.

| column | meaning |
|---|---|
| `rekon` | `1` = the position reconstructs from the codes alone, `0` = free |
| `L1 L2 L3` | `1` = changing a single code at this level moves this position |
| `zmien_na` | the base to change the input to under P1; `.` = no change |
| `wagaP` | normalised gradient magnitude |

A position with `rekon = 1` and zeros in all layers will be
overwritten by the decoder. The `kompakt` key holds the same data as
parallel arrays, ready for `pandas.DataFrame`.

In [ ]:
code_, m = call('/nawigator/mapa', {'sekwencja': WILD, 'od': 0, 'ile': 800})
assert code_ == 200, m
print('species        :', m['gatunek'])
print('reconstructs   : %.4f (does not reconstruct %d of 800)'
      % (m['rekon_frakcja'], m['nie_rekonstruuje']))
print('changes und. P1:', m['zmian_pod_gatunek'],
      '| reconstruction error:', m['blad_odtworzenia'], '<- these are NOT tips')
print('layers         :', m['rozklad_warstw'])

recs = [w for w in m['pozycje'] if w['zmien_na'] != '.']
free = [w['poz'] for w in m['pozycje'] if w['rekon'] == 0]
overwritten = [w['poz'] for w in m['pozycje']
               if w['rekon'] == 1 and sum(w['warstwy']) == 0]
print()
print('RECOMMENDATIONS for %s: %d' % (m['gatunek'], len(recs)))
for w in recs:
    print('   position %3d: %s -> %s  (layers %s)'
          % (w['poz'], w['wej'], w['zmien_na'], w['warstwy']))
print('FREE %d, first 15: %s' % (len(free), free[:15]))
print('OVERWRITTEN %d, first 15: %s' % (len(overwritten), overwritten[:15]))

# import pandas as pd; pd.DataFrame(m['kompakt']).head(20)

## 5.4 Judge: pairwise verdict

`POST /sedzia` returns `silniejsza` (the stronger one) and `silniejsza_idx` (`0` or `1`).

In [ ]:
candidate = list(WILD)
for w in recs:
    candidate[w['poz'] - 1] = w['zmien_na']
candidate = ''.join(candidate)
print('bases changed:', sum(1 for a, b in zip(WILD, candidate) if a != b))

code_, s = call('/sedzia', {'a': WILD, 'b': candidate,
                            'nazwa_a': 'wild', 'nazwa_b': 'candidate'})
print(code_, s)
print('CANDIDATE won' if s.get('silniejsza_idx') == 1
      else 'WILD came back -- no improvement shown (may be a tie)')

## 5.5 The set of 100 promoters

The task comes with the file **`Promotory.csv`**: 100 natural
promoters from 19 *Trichoderma* species and strains, each 800 bp, with no `N`.
The separator is a semicolon.

| column | meaning |
|---|---|
| `nazwa` | gene identifier in the source genome |
| `gatunek`, `gatunek_krotko` | species or strain of origin |
| `genom` | genome code in the set |
| `dlugosc`, `N` | 800 and 0 for all rows |
| `sekwencja` | the promoter, aligned to the transcription start site |

**What it is for.** To look at what real promoters of this genus of fungi
look like: which positions the Navigator considers free in them, where it sees
levers, how they fare in Judge comparisons against your variants.

**NOTE:** this is not a set of good answers.

In [ ]:
import csv

# Make your own path
with open('/content/Promotory.csv', encoding='utf-8') as fh:
    DATASET = list(csv.DictReader(fh, delimiter=';'))

print('sequences:', len(DATASET))
species = {}
for w in DATASET:
    species[w['gatunek_krotko']] = species.get(w['gatunek_krotko'], 0) + 1
print('species  :', len(species))
for g in sorted(species):
    print('   %-28s %d' % (g, species[g]))

SEQUENCES = [w['sekwencja'] for w in DATASET]
assert all(len(s) == 800 for s in SEQUENCES)

code_, o = call('/sedzia', {'a': WILD, 'b': SEQUENCES[0],
                            'nazwa_a': 'wild', 'nazwa_b': DATASET[0]['nazwa']})
print()
print('stronger:', o['silniejsza'], '| idx:', o['silniejsza_idx'])

## 5.6 Navigator: proposed edits

`POST /nawigator/edycje` changes latent codes and returns the resulting variants.
The `zmiany` field is counted against the baseline, that is, decoding without
any change to the codes.

| level | layer | slots | bp/slot | alphabet |
|---|---|---|---|---|
| 0 | L1 | 50 | 16 | 4 |
| 1 | L2 | 200 | 4 | 8 |
| 2 | L3 | 400 | 2 | 4 |

In [ ]:
CANDIDATES = 100     # WE AIM FOR 100 -- that is what the Oracle scores

code_, e = call('/nawigator/edycje', {'sekwencja': WILD, 'poziom': 2,
                                      'ile_kodow': 8, 'opcji': 8})
assert code_ == 200, e
print('layer %s | slots %d | alphabet %d | reconstruction error %d'
      % (e['warstwa'], e['slotow'], e['alfabet'],
         e['blad_rekonstrukcji_pozycji']))

pool = {candidate: 'from_map'}
for o in e['opcje']:
    pool[o['sekwencja']] = 'edits_%d' % o['nr']

# A few rounds to gather a bigger pool -- the Navigator limit is 600/min.
for rnd in range(2, 30):
    if len(pool) >= CANDIDATES:
        break
    code_, e2 = call('/nawigator/edycje',
                     {'sekwencja': WILD, 'poziom': 2, 'ile_kodow': 6 + rnd,
                      'opcji': 8, 'ziarno': 1000 + rnd})
    if code_ != 200:
        print('round %d: HTTP %s' % (rnd, code_)); break
    for o in e2['opcje']:
        pool[o['sekwencja']] = 'r%d_%d' % (rnd, o['nr'])
print('pool of unique candidates:', len(pool))

## 5.7 Comparing variants with the starting sequence

The Judge has no interval between calls, so you can check
arbitrarily many pairs within the per-minute limit.

In [ ]:
winners = []
for i, (seq, label) in enumerate(list(pool.items())[:25], 1):
    code_, s = call('/sedzia', {'a': WILD, 'b': seq,
                                'nazwa_a': 'wild', 'nazwa_b': label})
    if code_ != 200:
        print('%s: HTTP %s' % (label, code_)); break
    if s['silniejsza_idx'] == 1:
        winners.append((label, seq))
print('beat the wild type: %d of %d checked (%.0f%%)'
      % (len(winners), i, 100.0 * len(winners) / max(i, 1)))
print('NOTE: we send the WHOLE pool (%d), not only the winners.' % len(pool))
for label, _ in winners[:10]:
    print('  ', label)

## 5.8 Uploading the FASTA file

`POST /wgraj`, once per 5 minutes. The response reports the filtering
statistics and the number of sequences that entered scoring.

In [ ]:
# Judge winners at the start of the file, then the rest of the pool.
# The first 100 sequences are scored.
LIMIT = 100
winning_seqs = {s_ for _, s_ in winners}
to_upload = ([(l_, s_) for l_, s_ in winners]
             + [(l_, s_) for s_, l_ in pool.items() if s_ not in winning_seqs])
if len(to_upload) > LIMIT:
    print('pool %d -> trimmed to %d (rules limit)'
          % (len(to_upload), LIMIT))
    to_upload = to_upload[:LIMIT]
lines = []
for label, seq in to_upload:
    lines += ['>' + label, seq]
fasta = '\n'.join(lines)
print('file: %d sequences, %d characters' % (len(to_upload), len(fasta)))

code_, r = call('/wgraj', {'fasta': fasta})
print(code_)
if code_ == 200:
    print('--- filtering ---')
    for k2, v in r['filtrowanie'].items():
        print('   %-26s %s' % (k2, v))
    print('--- result ---')
    for k2 in ('ocenionych', 'dzielnik_top10', 'dzielnik_top100',
               'pozycja_top10', 'pozycja_top100', 'punkty_razem'):
        print('   %-26s %s' % (k2, r.get(k2)))
else:
    print(r)

## 5.9 Points table

`GET /ranking`, no limit. Only the points are visible.

In [ ]:
code_, t = call('/ranking')
assert code_ == 200, t
print('teams %d | competing %d | your position: %s'
      % (t['n_druzyn'], t['n_startujacych'], t['twoja_pozycja']))
print('tie:', t['remis'])
print()
print('%-4s %-13s %-6s %-8s %-9s %-7s %s'
      % ('pos', 'team', 'scored', 'TOP10', 'TOP100', 'total', 'upload'))
print('-' * 74)
for x in t['ranking']:
    print('%-4d %-13s %-6d %-8s %-9s %-7s %s'
          % (x['pozycja'], x['druzyna'], x['ocenionych'],
             x['punkty_top10'], x['punkty_top100'], x['punkty_razem'],
             x['wgranie_o'] or '-'))

## 5.10 Error codes

| code | meaning |
|---|---|
| 401 | missing `X-API-Key` |
| 403 | key unknown, expired, or endpoint is jury-only |
| 422 | sequence is not 800 bp, has characters outside `ACGTN`, or above 10% `N` |
| 429 | per-minute limit, team daily limit, or 5 minutes since the last upload |
| 503 | GPU queue full, the client retries automatically |
| `error code: 1010` | missing `User-Agent`; a Cloudflare response, not the API |

The model predicts strength from expression data and was not validated
in the laboratory, so the result is not a prediction of wet-lab activity.